Problem 1

In [ ]:
import numpy as np
import pandas as pd
import skfuzzy as fuzz
from sklearn.preprocessing import MinMaxScaler
np.random.seed(42)
n=1000

data=pd.DataFrame({
"Voltage":np.random.uniform(210,250,n),
"Current":np.random.uniform(5,120,n),
"Frequency":np.random.uniform(49,51,n),
"Load":np.random.uniform(20,100,n),
"PowerFactor":np.random.uniform(0.7,1,n),
"Temperature":np.random.uniform(20,80,n)
})

print(data.head())

      Voltage     Current  Frequency       Load  PowerFactor  Temperature
0  224.981605   26.290287  49.523411  73.816240     0.871599    43.618131
1  248.028572   67.318609  49.493958  83.734512     0.941630    48.406140
2  239.279758  105.388771  50.812509  40.037432     0.928048    71.272844
3  233.946339   89.205862  49.499092  69.989928     0.746170    40.400263
4  216.240746   97.754532  49.543899  65.739679     0.744775    72.178981


In [ ]:
!pip install scikit-fuzzy


In [2]:
data.to_csv("problem1_dataset.csv")

In [8]:

scaler=MinMaxScaler()
X=scaler.fit_transform(data)

cntr,u,u0,d,jm,p,fpc=fuzz.cluster.cmeans(
X.T,
c=4,
m=2,
error=0.005,
maxiter=1000,
init=None
)
labels=np.argmax(u,axis=0)
print("FPC:(Quality of clustering it shows):")
print(fpc)

centers=pd.DataFrame(
scaler.inverse_transform(cntr),
columns=data.columns
)
print("Cluster centres:")
print(centers)

membership=pd.DataFrame(
u.T,
columns=["Normal","Overload","Voltage","Frequency"]
)
print("Sample of membership matrix:")
print(membership.head())

FPC:(Quality of clustering it shows):
0.2500001143128244
Cluster centres:
      Voltage    Current  Frequency       Load  PowerFactor  Temperature
0  229.603456  63.295992  50.004750  59.226314     0.848215    49.910634
1  229.629104  63.307326  50.004340  59.231843     0.848288    49.898740
2  229.613155  63.321978  50.005192  59.217500     0.848294    49.887104
3  229.595334  63.302660  50.004964  59.244356     0.848129    49.934999
Sample of membership matrix:
     Normal  Overload   Voltage  Frequency
0  0.250090  0.250079  0.249841   0.249990
1  0.249887  0.250369  0.250016   0.249728
2  0.249938  0.250008  0.250091   0.249963
3  0.249952  0.250132  0.249917   0.249999
4  0.250065  0.249786  0.249766   0.250383


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca=PCA(n_components=2)
X2=pca.fit_transform(X)

plt.scatter(
X2[:,0],
X2[:,1],
c=labels,
cmap="viridis"
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


###Problem 2

In [16]:
import random
import pandas as pd

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

random.seed(42)

appliances = [
    "AC",
    "Fan",
    "Light",
    "TV",
    "Refrigerator",
    "Washing Machine",
    "Microwave",
    "Laptop",
    "Water Heater",
    "Air Purifier"
]

transactions = []

for _ in range(1000):
    n = random.randint(2, 6)
    transaction = random.sample(appliances, n)
    transactions.append(transaction)

print("Sample Transactions:")
for t in transactions[:5]:
    print(t)

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df = pd.DataFrame(te_array, columns=te.columns_)

frequent_itemsets = apriori(df, min_support=0.1, use_colnames=True)

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.2
)

print("\nFrequent Itemsets")
print(frequent_itemsets)

print("\nAssociation Rules")
print(rules[["antecedents","consequents","support","confidence","lift"]])

print("\nEnergy Saving Recommendations")

for _, row in rules.iterrows():
    a = ", ".join(row["antecedents"])
    b = ", ".join(row["consequents"])
    print(f"If {a} is ON, {b} is usually ON. Consider switching OFF {a} and {b} together when not required.")

Sample Transactions:
['AC', 'Refrigerator']
['TV', 'Light', 'Fan']
['Fan', 'Microwave', 'AC', 'Laptop', 'Water Heater', 'Air Purifier']
['Water Heater', 'AC', 'TV']
['Microwave', 'TV', 'Laptop', 'Refrigerator', 'Light', 'AC']

Frequent Itemsets
    support                         itemsets
0     0.387                             (AC)
1     0.399                   (Air Purifier)
2     0.409                            (Fan)
3     0.401                         (Laptop)
4     0.395                          (Light)
5     0.411                      (Microwave)
6     0.379                   (Refrigerator)
7     0.395                             (TV)
8     0.399                (Washing Machine)
9     0.417                   (Water Heater)
10    0.171               (Air Purifier, AC)
11    0.150                        (Fan, AC)
12    0.146                     (AC, Laptop)
13    0.134                      (Light, AC)
14    0.137                  (Microwave, AC)
15    0.144               (AC, Refr

In [13]:
df = pd.DataFrame({
    "Transaction_ID": range(1, len(transactions) + 1),
    "Appliances": [", ".join(t) for t in transactions]
})

df.to_csv("home_appliance_transactions.csv", index=False)